# B1 · A forest on the Titanic driven by the OOB error

**Type**: Implement with scikit-learn · **Level**: ●●○ independent · **Time**: about 50 minutes · **core path ★**

## Objective

Build a pipeline with a `RandomForestClassifier` on the Titanic and use the **out-of-bag** error instead of cross-validation to: see how many trees are needed (OOB curve with `warm_start`), choose `max_features` and `min_samples_leaf`, and compare the OOB estimate with the cross-validation one and with the test set.

## Dataset

`ds.titanic()`, stratified 75/25 split: `X_tr, X_te, y_tr, y_te`. Preprocessing (given in `build_prep()`): numeric `pclass, age, siblings_spouses, parents_children, fare` (median) and categorical `sex, port, deck, title` (most frequent + `OrdinalEncoder`).

## Educational purpose

Learn to use the OOB as a working tool: it is free (no retraining per fold), it is honest (it uses only trees that have not seen the example) and it lets you iterate quickly on the hyperparameters.

## Task

1. With `warm_start=True` train the pipeline increasing `n_estimators` in `[5, 10, 20, 50, 100, 200, 400]` and save `oob_curve` (`pd.Series` of the OOB **error** = `1 − oob_score_`, index: number of trees) and `stable_n`: the minimum number of trees for which the OOB error is within **0.005** of the final value (at 400).
2. With `n_estimators = 300` compute the OOB error for `max_features` in `[1, 2, 3, "sqrt", 5, 9]` (`mf_grid`, `pd.Series` indexed by the label as a string) and for `min_samples_leaf` in `[1, 2, 5, 10]` (`leaf_grid`); save `best_mf` and `best_leaf`.
3. Train the model with the best values: save `oob_err` (OOB error), `cv_err` (error in `StratifiedKFold(5, shuffle=True, random_state=0)` on the training set) and `test_err` (error on the test set). Compare them: which one is the most optimistic?
4. Save `train_acc` (accuracy on the training set) and show how misleading it is.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder

df = ds.titanic()
X, y = df.drop(columns="survived"), df["survived"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
cv = StratifiedKFold(5, shuffle=True, random_state=0)


def build_prep():
    return ColumnTransformer([("num", SimpleImputer(strategy="median"), ["pclass", "age", "siblings_spouses", "parents_children", "fare"]),
                              ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)), ["sex", "port", "deck", "title"])])


def model(**kw):
    return make_pipeline(build_prep(), RandomForestClassifier(oob_score=True, random_state=0, n_jobs=-1, **kw))

## Your work

In [ ]:
oob_curve = pd.Series(dtype=float)
stable_n = None

In [ ]:
mf_grid = pd.Series(dtype=float)
leaf_grid = pd.Series(dtype=float)
best_mf = None
best_leaf = None

In [ ]:
oob_err = None
cv_err = None
test_err = None
train_acc = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`m = model(warm_start=True)`; in the loop `m.set_params(randomforestclassifier__n_estimators=n).fit(X_tr, y_tr)`; `1 - m[-1].oob_score_`. For `max_features`: `model(n_estimators=300, max_features=mf)`. CV: `cross_val_score(model(...), X_tr, y_tr, cv=cv).mean()`.'):
    check(list(oob_curve.index) == [5, 10, 20, 50, 100, 200, 400] and oob_curve[5] > oob_curve[100] + 0.02 and 20 <= stable_n <= 200, "the OOB error drops from 5 to 100 trees by over 2 points and settles between 20 and 200 trees", "warm_start=True and set_params(randomforestclassifier__n_estimators=n)")
    check(len(mf_grid) == 6 and len(leaf_grid) == 4 and str(best_mf) in mf_grid.index and best_leaf in (1, 2, 5, 10), "six values of max_features and four of min_samples_leaf", "one OOB error for each value")
    check(mf_grid["1"] > mf_grid.min() + 0.01, "with max_features=1 the error is worse than the best by over 1 point (trees too weak)", "compare the OOB errors")
    check(0.12 < oob_err < 0.25 and 0.12 < cv_err < 0.25 and 0.12 < test_err < 0.25, "OOB, cross-validation and test errors between 0.12 and 0.25 (accuracies between 75 and 88%)", "1 - accuracy")
    check(abs(oob_err - cv_err) < 0.03 and abs(oob_err - test_err) < 0.05, "the OOB error is close to the cross-validation one (within 3 points) and to the test one (within 5)", "OOB as an honest estimate")
    check(train_acc > 0.85 and 1 - train_acc < oob_err * 0.85, "on the training set the accuracy goes over 0.85: misleading (error clearly below the OOB one)", "model.score(X_tr, y_tr)")
done()

---
The commented solution is in `solutions/B_with_sklearn/B1_forest_on_the_titanic_with_oob.ipynb`. Open it after you pass the check, or after you have really tried.